# Advanced Wi-Fi Track Data Clean

This method re-clean data according to track repeat frequency ( caused by unstable signal appears among multiple trackers) based on given basic cleaned data.

Generate **virtual wifi tracker** after cleaning.(Virtual wifi trackers' id will be under zero)

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mat
import matplotlib.cm as cm
import os
import datetime
import time
from tqdm import tqdm
import plotly.express as px
import plotly.io as pio
import seaborn as sns

import sys
sys.path.append('../plot/')
import myplot

sys.path.append('../utils/')
import utils
import DBSCAN
from DBSCAN import My_DBSCAN,My_DBSCAN_MATRIX

# 设置plotly默认主题
pio.templates.default = 'plotly_white'

mat.rcParams['font.family'] = 'SimHei'
mat.rcParams['font.sans-serif'] = 'SimHei'

import warnings
warnings.filterwarnings('ignore')

from collections import namedtuple

from importlib import reload

## 读取数据->初步清理->保存

In [2]:
df_read = pd.read_csv(os.getcwd()+'dacang/cleaned_data/dacang_cleaned_data_basic.csv')
df = df_read.copy()
df_read

,a,r,t,m
0,22,-43,2022-07-05 23:49:36,"48,74,38,155,214,98"
1,22,-35,2022-07-05 23:49:36,"252,107,240,89,142,53"
2,22,-42,2022-07-05 23:49:36,"164,125,159,153,152,106"
3,22,-38,2022-07-05 23:49:47,"252,107,240,89,142,53"
4,22,-42,2022-07-05 23:49:47,"164,125,159,153,152,106"
...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"88,102,186,101,140,144"
1916097,66,-49,2022-08-07 23:46:54,"88,102,186,101,140,144"
1916098,66,-48,2022-08-07 23:48:38,"88,102,186,101,140,144"
1916099,66,-49,2022-08-07 23:51:14,"88,102,186,101,140,144"


In [3]:
#label each mac with date
pbar = tqdm(total=len(df))
df['oriMac'] = df['m']
df.t = pd.to_datetime(df.t)
df['calender'] = df.t.apply(lambda x : str(x.month)+'.'+str(x.day))
for index,row in df.iterrows():
    df.loc[index,'m'] = str(row['calender'])+'-'+row.m
    pbar.update(1)
pbar.close()
print("mac length:",len(df.m.unique()))
df

  0%|          | 0/1916101 [00:00<?, ?it/s]

100%|██████████| 1916101/1916101 [02:24<00:00, 13217.61it/s]


,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


In [5]:
#删除打上标签后只出现过一次的mac
df = df[df.duplicated('m',keep=False)]
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


In [ ]:
#保存
df.to_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_data_dataLabeled.csv",index=False)

## 读取已打上date标签的dataframe

In [2]:
df = pd.read_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_data_dataLabeled.csv")
df.t = pd.to_datetime(df.t)
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1899913,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899914,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899915,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899916,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


## 以天为单位清理异常mac

In [68]:
#获取每个mac一天中经过的探针数：a_count
count_list = df.groupby('m').a.value_counts()
df_count = count_list.to_frame().reset_index()
a_count = df_count.m.value_counts()
#获取每个mac一天中的数据量:s_count
count_list = df.m.value_counts()


mac_list = df.m.unique()
len(mac_list)

15615

In [84]:
def GetDuration(df_now):
    t1 = df_now.iloc[0].t
    t2 = df_now.iloc[len(df_now)-1].t
    duration =  t2-t1
    t = (t1 + duration/2)
    moment = t.hour + (t.minute/60)
    duration = duration.components.hours+(duration.components.minutes/60)
    return round(duration,2),round(moment,2)

macDur_dict = {}
macMoment_dict = {}

for mac in tqdm(mac_list):
    df_now = df[df.m == mac]
    duration,moment = GetDuration(df_now)
    macDur_dict.update({mac:duration})
    macMoment_dict.update({mac:moment})

#保存
dur_dict_str = str(macDur_dict)
mom_dict_str = str(macMoment_dict)
with open('dacang/cluster/date_mac_duration_dict.txt', 'w') as file:
   file.write(dur_dict_str)
with open('dacang/cluster/date_mac_moment_dict.txt', 'w') as file:
   file.write(mom_dict_str)

100%|██████████| 15615/15615 [20:13<00:00, 12.86it/s]


In [86]:
#读取
with open('dacang/cluster/date_mac_duration_dict.txt', 'r') as file:
    dict_str = file.read()
    macDur_dict = eval(dict_str)
with open('dacang/cluster/date_mac_moment_dict.txt', 'r') as file:
    dict_str = file.read()
    macMoment_dict = eval(dict_str)
mac_list = list(macDur_dict.keys())
mac_dur_list = list(macDur_dict.values())
mac_moment_list = list(macMoment_dict.values())

In [108]:
#生成统计数值的dataframe
df_sta = pd.DataFrame({'M':[],'Dur':[],'A_count':[],'Count':[],'oriMac':[],'date':[]})
for i in range(len(mac_list)):
    df_sta = df_sta._append({'M':mac_list[i],
                             'Dur':mac_dur_list[i],
                             'Moment':mac_moment_list[i],
                             'A_count':a_count[mac_list[i]],
                             'Count':count_list[mac_list[i]],
                             'oriMac':mac_list[i].split('-')[1],
                             'date':mac_list[i].split('-')[0]},ignore_index=True)
df_sta

,M,Dur,A_count,Count,oriMac,date,Moment
0,"7.5-48,74,38,155,214,98",21.83,6.0,147.0,"48,74,38,155,214,98",7.5,22.73
1,"7.5-252,107,240,89,142,53",17.02,2.0,37.0,"252,107,240,89,142,53",7.5,20.33
2,"7.5-164,125,159,153,152,106",17.02,2.0,19.0,"164,125,159,153,152,106",7.5,20.33
3,"7.5-112,138,9,131,175,199",17.02,2.0,47.0,"112,138,9,131,175,199",7.5,20.33
4,"7.5-208,151,254,40,222,169",17.02,3.0,11.0,"208,151,254,40,222,169",7.5,20.33
...,...,...,...,...,...,...,...
15610,"8.7-172,92,102,108,72,204",0.00,1.0,2.0,"172,92,102,108,72,204",8.7,19.02
15611,"8.7-172,186,190,231,156,25",0.00,1.0,2.0,"172,186,190,231,156,25",8.7,19.02
15612,"8.7-216,138,220,202,211,84",1.93,1.0,8.0,"216,138,220,202,211,84",8.7,22.82
15613,"8.7-244,76,112,14,81,63",0.03,1.0,3.0,"244,76,112,14,81,63",8.7,21.88


In [109]:
fig = px.histogram(df_sta.Count.to_list())
fig.show()

In [110]:
#--初步清理--#
#剔除一天中数据量小于10且大于10000的mac
df_sta = df_sta[df_sta.Count.apply(lambda x : x>10 and x<10000)]
#剔除一天中持续时间小于半小时的mac
df_sta = df_sta[df_sta.Dur>0.5]
#剔除一天中接受探针数小于3的mac
df_sta = df_sta[df_sta.A_count>3]
df_sta = df_sta.reset_index().drop('index',axis = 1)
df_sta

,M,Dur,A_count,Count,oriMac,date,Moment
0,"7.5-48,74,38,155,214,98",21.83,6.0,147.0,"48,74,38,155,214,98",7.5,22.73
1,"7.5-20,178,229,137,47,142",17.02,4.0,38.0,"20,178,229,137,47,142",7.5,20.33
2,"7.6-48,74,38,155,214,98",19.23,12.0,8186.0,"48,74,38,155,214,98",7.6,9.62
3,"7.6-20,178,229,137,47,142",18.90,5.0,2634.0,"20,178,229,137,47,142",7.6,9.45
4,"7.6-4,207,140,11,128,186",23.98,4.0,947.0,"4,207,140,11,128,186",7.6,12.00
...,...,...,...,...,...,...,...
1344,"8.7-176,70,146,156,89,69",9.38,5.0,179.0,"176,70,146,156,89,69",8.7,17.18
1345,"8.7-44,220,173,148,169,213",7.48,4.0,171.0,"44,220,173,148,169,213",8.7,17.82
1346,"8.7-124,161,119,201,50,40",9.78,6.0,260.0,"124,161,119,201,50,40",8.7,18.98
1347,"8.7-152,207,83,118,160,172",19.45,4.0,54.0,"152,207,83,118,160,172",8.7,12.45


## 清除离群值 - 随机孤立森林

In [111]:
#生成用于处理利群数据的dataframe、并归一化
df_outlier = utils.Normalize_df(df_sta.reindex(columns = ['Dur','A_count','Count','Moment']))
print(len(df_outlier))
df_outlier.head()

1349


,Dur,A_count,Count,Moment
0,9.083546,1.666667,0.137805,9.575552
1,7.033248,0.000000,0.027358,8.556876
2,7.975277,6.666667,8.283514,4.011036
3,7.834612,0.833333,2.657817,3.938879
4,10.000000,0.000000,0.948424,5.021222


In [112]:
reload(myplot)
myplot.Boxes([(df_outlier.Dur,"数据持续时间"),(df_outlier.Count,"数据量"),
              (df_outlier.A_count,"探针数量"),(df_outlier.Moment,"中位时刻")])

所经过的探针数量多不应当被视为噪声，因此在排除离群值时仅考虑数据持续时间和数据量

In [113]:
from sklearn.ensemble import IsolationForest
rng = np.random.RandomState(42)
model_IF = IsolationForest(max_samples=100,random_state=rng,contamination='auto')
model_IF.fit(df_outlier[['Dur','Count']])
df_outlier['label'] = model_IF.predict(df_outlier[['Dur','Count']])
print(df_outlier.label.value_counts())

fig = px.scatter(df_outlier,y = 'Count',x = 'Dur',color='label')
fig.update_traces(marker_size = 5)
fig.update_layout(scattermode='group')
fig.show()

label
 1    927
-1    422
Name: count, dtype: int64


In [114]:
df_cluster = df_outlier[df_outlier.label == 1].reset_index().drop('index',axis = 1)
df_cluster = utils.Normalize_df(df_cluster,cols = ['Dur','Moment','Count'])
myplot.Scatter_3D(df_cluster,'Dur','Count','Moment','label',color_name='A_count')

**偶发信号**和和**多探针覆盖范围下的智能设备信号**是需要清洗的内容，
- 偶发信号的特征为数据量极少且到达探针的数量少
- 多探针覆盖范围下的智能设备信号特征为数据量多且到达探针的数量为1-4个


另外主要还有**常驻人群**，**游客**，**通勤人群**三类数据，但由于信号干扰等原因，不同数据之间的分隔基准并不明确，同时人群种类可能并不仅仅只有前文所列的三类，不同类型也有可能因为信号干扰、行为模式不同而导致数据差异，因此通过简单的逻辑判断清洗效果和分类效果都不甚理想。

尝试通过kmeans或dbscan聚类来进行分类

## DBSCAN聚类
DBSCAN（Density-Based Spatial Clustering of Applications with Noise）是一种基于密度的聚类算法，可以将数据点分成不同的簇，并且能够识别噪声点（不属于任何簇的点）。

DBSCAN聚类算法的基本思想是：

在给定的数据集中，根据每个数据点周围其他数据点的密度情况，将数据点分为核心点、边界点和噪声点。

- 核心点是周围某个半径内有足够多其他数据点的数据点；
- 边界点是不满足核心点要求，但在某个核心点的半径内的数据点；
- 噪声点则是不满足任何条件的点。


## 轮廓系数、卡林斯基-哈拉巴斯指数、DBCV进行评估

- **轮廓系数（silhouette_score）**，轮廓系数指标不关注样本的实际类别，而是通过分析聚类结果中样本的内聚度和分离度两种因素来给出成绩，取值范围为（-1，1），**值越大代表聚类的结果越合理**。
- **卡林斯基-哈拉巴斯指数(Calinski-Harabasz Index)**，组内离散程度低，**协方差矩阵**的迹就会越小，同时，组间离散程度大，**协方差矩阵**的迹就会越大。因此calinski_harabasz**指数越高越好**。
  
 $$ s(k)=\frac{T_r(B_k)}{T_r(W_k)}*\frac{N-k}{k-1} $$

其中$N$是数据集的样本量，$k$为簇的个数，$B_k$是组间离散矩阵,即不同簇之间的协方差矩阵。$W_k$是簇内离散矩阵，即一个簇内数据的协方差矩阵，而$Tr$表示矩阵的迹。


In [54]:
#多次聚类，确定效果最佳的eps与min_samples
results = My_DBSCAN_MATRIX(df_cluster,
                 init_eps = 0.13,
                 step_eps = 0.01,
                 epoch_eps = 1,
                 init_mSamples = 3,
                 step_mSamples = 1,
                 epoch_mSamples = 1
                 )
            

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  6.83it/s]


In [20]:
#保存聚类数据
df_dbscan_result = pd.DataFrame({
    "cluster_num":list(map(lambda x : x.cluster_num, results)),
    "noise_num":list(map(lambda x : x.noise_num, results)),
    "dbcv":list(map(lambda x : x.dbcv, results)),
    "silhouette":list(map(lambda x : x.silhouette, results)),
    "calinski":list(map(lambda x : x.calinski, results)),
    "eps":list(map(lambda x : x.eps, results)),
    "min_samples":list(map(lambda x : x.min_samples, results)),
    })

df_dbscan_result.to_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise_1130.csv",index=False)
#np.save(os.getcwd()+"/dacang/cluster/cluster_mat_result_1128.npy",cluster_labels)

## 聚类结果评估

In [75]:
def Cut_Data_By_NoiseNum(df,cut_thre):
    df_result = df.copy()
    for index,row in df_result.iterrows():
        if row.noise_num > cut_thre :
            df_result.at[index,'dbcv'] = -1
            df_result.at[index,'cluster_count'] = 0
            df_result.at[index,'silhouette'] = -1
            df_result.at[index,'calinski_harabasz'] = 0
            df_result.at[index,'noise_num'] = 0
    return df_result
        

In [79]:
def add_data(df,col_name,list):
    min_samples_unique = df.min_samples.unique()
    df_3d = pd.DataFrame(columns=min_samples_unique)
    df_3d.insert(0,'eps',[])
    eps = df.eps.unique()
    min_samples = df.min_samples
    for i in range(len(eps)):
        df_now = (df[df.eps == eps[i]])
        add_dict = {"eps":eps[i]}
        for index,row in df_now.iterrows():
            add_dict.update({row.min_samples:row[col_name]})
        
        df_3d = df_3d._append(add_dict,ignore_index=True)

    Sur_Data = namedtuple("Sur_Data",['values','xAxes','yAxes','name'])
    data = Sur_Data(
                    values=df_3d.drop('eps',axis=1).values,
                    xAxes=df_3d.columns[1:],
                    yAxes=df_3d.eps,
                    name=col_name)
    list.append(data)
    
    # myplot.Surface3D(df_3d.drop('eps',axis=1).values,df_3d.eps,df_3d.columns[1:],
    #                 x_name = 'eps',y_name = "min_samples")

In [80]:
def ShowResult(df,col_name_list):
    data_list = []
    for name in col_name_list:
        add_data(df,name,data_list)
    myplot.Surface3D_supPlot(data_list)

### 第一次聚类结果评估

In [81]:
#读取聚类数据
df_dbscan_result = pd.read_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise.csv")
cluster_labels =  np.load(os.getcwd()+"/dacang/cluster/cluster_mat_result.npy")
df_result = Cut_Data_By_NoiseNum(df_dbscan_result,800)

In [85]:
ShowResult(df_dbscan_result,['dbcv','noise_num'])

### 第二次聚类数据评估

In [18]:
#读取聚类数据
df_dbscan_result = pd.read_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise_1128.csv")
cluster_labels =  np.load(os.getcwd()+"/dacang/cluster/cluster_mat_result_1128.npy")

In [33]:
df_result = Cut_Data_By_NoiseNum(df_dbscan_result,800)

In [19]:
ShowResult(df_dbscan_result,['dbcv','noise_num','cluster_count','silhouette','calinski_harabasz'])

### 第三次聚类结果评估

In [6]:
df_dbscan_result = pd.read_csv(os.getcwd()+"/dacang/cluster/dbscan_result_eliNoise_1129.csv")
ShowResult(df_dbscan_result,['dbcv','noise_num','cluster_count'])

## 针对第一次评估的断崖进行分类测试
eps = 0.74

In [46]:
df_dbscan_now = df[df.min_samples == set(df.min_samples)[1]]
myplot.Double_Axes_Line(df_dbscan_now,"eps","silhouette","calinski_harabasz",
                        xAxes_name = "领域半径(eps)",
                        line1_name = "轮廓系数",
                        line2_name = "卡林斯基-哈拉巴斯指数")

轮廓系数与卡林斯基-哈拉巴斯指数都是内部聚类评价指标，即在不知道真实labels时对聚类结果进行评价。

内部聚类评价指标的一个通病是默认数据集中的类别是簇状结构。

事实上，真实世界的数据集很多都不是簇状结构，所以内部聚类评价往往并不客观。

对于非簇状结构的聚类结果的评价往往使用外部聚类评价指标，常见的有NMI，AMI，F-score，Accuracy等。

In [47]:
#聚类数量折线图
df_dbscan_now = df[df.min_samples == set(df.min_samples)[1]]
myplot.One_Axes_Line(df_dbscan_now,"eps","cluster_count",
                     xAxes_name = "领域半径(eps)",
                     line_name = "聚类数量")

In [40]:
#噪声数量
df_dbscan_now = df[df.min_samples == set(df.min_samples)[1]]
myplot.One_Axes_Line(df_dbscan_now,"eps","noise_num",
                     xAxes_name = "领域半径(eps)",
                     line_name = "噪声数量")

## 确定eps和min_samples后分析

In [ ]:
#eps = 16 , min_samples = 0.73


## test
-----

In [88]:
from sklearn.cluster import DBSCAN
#if 'label' is in df_cluster.col
df_cluster = df_cluster.drop('label',axis = 1)
dbscan = DBSCAN(eps=0.62, min_samples=10)
cluster = dbscan.fit(df_cluster)
len(set(cluster.labels_))
df_cluster['label'] = cluster.labels_

In [73]:
from collections import Counter
Counter(cluster.labels_)

Counter({0: 213,
         1: 589,
         2: 411,
         -1: 94,
         3: 243,
         4: 18,
         5: 89,
         6: 148,
         7: 27,
         8: 46,
         9: 15,
         10: 8,
         12: 7,
         11: 12,
         13: 9,
         14: 7,
         15: 8,
         16: 6})

In [89]:
myplot.Date_Mac_3D_scatter(df_cluster,'Dur','Count','A_count','label')